# 🛡️ RAKSHAK-ICS — Exploratory Data Analysis

**Project:** RAKSHAK-ICS · Resilient Anomaly Knowledge System for Hardening Industrial Control Systems  
**Dataset:** SWaT A9 (Secure Water Treatment — Clean / Normal Operations)  
**HAI Dataset:** HIL-based Augmented ICS Security Dataset  

---

> ⚠️ **NDA Notice:** The SWaT dataset is protected under NDA. This notebook displays **only aggregated/normalised statistics** — no raw sensor values are exposed.

---

### Notebook Sections
1. Setup & Imports
2. SWaT Dataset Overview
3. Missing Values & Bad Input Analysis
4. Statistical Summary (Aggregated)
5. Sensor Value Distributions
6. Time Series Visualization
7. Actuator State Analysis
8. Sensor Correlation Heatmap
9. Top Correlated Sensor Pairs
10. Process State Distribution
11. HAI Dataset Overview
12. Key Findings Summary

In [ ]:
# ─── Section 1: Setup & Imports ──────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from itertools import combinations

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Navigate to project root (notebook lives in notebooks/) ---
os.chdir('..')
print(f'Working directory: {os.getcwd()}')

# --- Dark theme setup ---
plt.style.use('dark_background')
%matplotlib inline

# Custom RAKSHAK colour palette
PALETTE = ['#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c084fc', '#ff922b']
sns.set_palette(PALETTE)

# Global plot defaults
plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor':   '#0e1117',
    'axes.edgecolor':   '#333333',
    'axes.labelsize':   12,
    'axes.titlesize':   16,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.dpi':       110,
    'savefig.dpi':      150,
})

print('✅ Setup complete — dark theme active')

---
## 2 · SWaT Dataset Overview

Load all three raw CSV files (`dataset1.csv`, `dataset2.csv`, `dataset3.csv`), concatenate them, and inspect shapes, dtypes, and column categories.

In [ ]:
# ─── Section 2: Load SWaT Data ──────────────────────────────────────
DATA_DIR = Path('data/swat')
csv_files = sorted(DATA_DIR.glob('dataset*.csv'))
print(f'Found {len(csv_files)} CSV files: {[f.name for f in csv_files]}\n')

# Load each file as string dtype first (handles mixed numeric + 'Bad Input')
dfs = []
for fp in csv_files:
    df_part = pd.read_csv(fp, dtype=str, na_values=[''], keep_default_na=True, low_memory=False)
    df_part.columns = df_part.columns.str.strip()
    df_part.dropna(how='all', inplace=True)
    print(f'  {fp.name}: {df_part.shape[0]:>7,} rows  × {df_part.shape[1]} cols')
    dfs.append(df_part)

df_raw = pd.concat(dfs, axis=0, ignore_index=True)
print(f'\n📦 Combined dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

In [ ]:
# ─── Column Classification ───────────────────────────────────────────
cols = list(df_raw.columns)

ts_cols     = [c for c in cols if c.lower() in ('t_stamp', 'timestamp')]
pv_cols     = [c for c in cols if c.endswith('.Pv')]
status_cols = [c for c in cols if c.endswith('.Status')]
alarm_cols  = [c for c in cols if c.endswith('.Alarm')]
speed_cols  = [c for c in cols if c.endswith('.Speed')]
state_cols  = [c for c in cols if c.startswith('P') and c.endswith('_STATE')]

classified = set(ts_cols + pv_cols + status_cols + alarm_cols + speed_cols + state_cols)
other_cols = [c for c in cols if c not in classified]

col_summary = pd.DataFrame({
    'Category': ['Timestamp', 'Continuous (.Pv)', 'Actuator (.Status)',
                 'Alarm (.Alarm)', 'Speed (.Speed)', 'State (P*_STATE)', 'Other'],
    'Count':    [len(ts_cols), len(pv_cols), len(status_cols),
                 len(alarm_cols), len(speed_cols), len(state_cols), len(other_cols)],
    'Examples': [
        ', '.join(ts_cols[:3]),
        ', '.join(pv_cols[:4]) + ' …',
        ', '.join(status_cols[:4]) + ' …',
        ', '.join(alarm_cols[:4]) + ' …',
        ', '.join(speed_cols[:3]),
        ', '.join(state_cols[:3]) + ' …',
        ', '.join(other_cols[:3]) if other_cols else '—',
    ],
})
print('\n🏷️  Column Classification')
print('=' * 80)
print(col_summary.to_string(index=False))
print(f'\nTotal columns: {len(cols)}')

---
## 3 · Missing Values & Bad Input Analysis

The SWaT dataset contains the string `"Bad Input"` in some sensor columns instead of numeric values.  
We count occurrences per column and visualise the missing-data pattern.

In [ ]:
# ─── Section 3: Bad Input & Missing Value Analysis ─────────────────
# Count 'Bad Input' occurrences per column (case-insensitive)
bad_input_counts = {}
for col in df_raw.columns:
    mask = df_raw[col].astype(str).str.strip().str.lower() == 'bad input'
    count = mask.sum()
    if count > 0:
        bad_input_counts[col] = count

if bad_input_counts:
    bi_df = pd.DataFrame({
        'Column': list(bad_input_counts.keys()),
        'Bad Input Count': list(bad_input_counts.values()),
        '% of Rows': [v / len(df_raw) * 100 for v in bad_input_counts.values()],
    }).sort_values('Bad Input Count', ascending=False).reset_index(drop=True)
    print(f'⚠️  {len(bad_input_counts)} columns contain "Bad Input" values:\n')
    print(bi_df.to_string(index=False))
else:
    print('✅ No "Bad Input" values detected in any column.')
    bi_df = pd.DataFrame()

In [ ]:
# ─── Missing-data heatmap ────────────────────────────────────────────
# Build a boolean matrix: True where value is NaN or 'Bad Input'
missing_matrix = df_raw.copy()
for col in missing_matrix.columns:
    is_bad = missing_matrix[col].astype(str).str.strip().str.lower() == 'bad input'
    is_na  = missing_matrix[col].isna()
    missing_matrix[col] = (is_bad | is_na).astype(int)

# Drop timestamp for the heatmap
if 't_stamp' in missing_matrix.columns:
    missing_matrix.drop(columns=['t_stamp'], inplace=True)

# Subsample rows for readable heatmap (every Nth row)
n_display_rows = 200
step = max(1, len(missing_matrix) // n_display_rows)
missing_sub = missing_matrix.iloc[::step]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(
    missing_sub.T, cmap=['#0e1117', '#ff6b6b'], cbar=False,
    yticklabels=True, xticklabels=False, linewidths=0, ax=ax,
)
ax.set_title('Missing / Bad Input Pattern  (red = missing or Bad Input)', fontsize=16, pad=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_xlabel(f'Rows (sampled every {step})', fontsize=12)
ax.tick_params(axis='y', labelsize=6)
plt.tight_layout()
plt.show()

---
## 4 · Statistical Summary (Aggregated — NDA Safe)

Descriptive statistics for **continuous `.Pv` sensors only**, after converting to numeric and dropping Bad Input.  
All values shown are aggregated — no individual sample data is revealed.

In [ ]:
# ─── Section 4: Statistical Summary ──────────────────────────────────
# Convert .Pv columns to numeric, coercing 'Bad Input' → NaN
df_numeric = df_raw[pv_cols].copy()
for col in df_numeric.columns:
    df_numeric[col] = pd.to_numeric(
        df_numeric[col].replace(r'(?i)^bad\s*input$', np.nan, regex=True),
        errors='coerce'
    )

# Normalise to [0, 1] for display — NDA safe
desc = df_numeric.describe().T
desc['non_null_%'] = (df_numeric.notna().sum() / len(df_numeric) * 100).round(2)
desc = desc[['count', 'non_null_%', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

print('📊 Aggregated Statistics — Continuous Sensors (.Pv)\n')
print(f'Showing {len(pv_cols)} sensor columns, {len(df_numeric):,} samples\n')
print(desc.round(3).to_string())

---
## 5 · Sensor Value Distributions

Histograms of six representative sensors spanning different sub-processes:

| Sensor | Sub-Process | Physical Meaning |
|--------|-------------|------------------|
| LIT101 | P1 — Raw Water | Tank level |
| FIT201 | P2 — Pre-treatment | Flow rate |
| AIT201 | P2 — Pre-treatment | Water quality (conductivity) |
| LIT301 | P3 — UF Filtration | UF feed tank level |
| FIT401 | P4 — De-chlorination | Flow rate |
| LIT601 | P6 — RO Backwash | Backwash tank level |

In [ ]:
# ─── Section 5: Sensor Distributions ─────────────────────────────────
sensor_picks = ['LIT101.Pv', 'FIT201.Pv', 'AIT201.Pv',
                'LIT301.Pv', 'FIT401.Pv', 'LIT601.Pv']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Sensor Value Distributions (Normalised Counts)',
             fontsize=18, fontweight='bold', y=1.02, color='white')

for idx, (sensor, ax) in enumerate(zip(sensor_picks, axes.flat)):
    colour = PALETTE[idx % len(PALETTE)]
    data = df_numeric[sensor].dropna()
    ax.hist(data, bins=60, color=colour, alpha=0.85, edgecolor='#1a1a2e', linewidth=0.5,
            density=True)
    ax.set_title(sensor, fontsize=14, fontweight='bold', color=colour)
    ax.set_xlabel('Sensor Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
    # Add vertical line at mean
    mean_val = data.mean()
    ax.axvline(mean_val, color='white', linewidth=1, linestyle='--', alpha=0.6)
    ax.text(mean_val, ax.get_ylim()[1] * 0.92, f' μ={mean_val:.1f}',
            color='white', fontsize=9, alpha=0.8)

plt.tight_layout()
plt.show()

---
## 6 · Time Series Visualization

Plotting four key sensors over time to inspect operational patterns, periodicity, and steady-state behaviour in the clean (normal) dataset.

In [ ]:
# ─── Section 6: Time Series ──────────────────────────────────────────
# Parse timestamps
ts = pd.to_datetime(df_raw['t_stamp'], format='%d-%b-%Y %H:%M:%S', errors='coerce')

ts_sensors = ['LIT101.Pv', 'FIT101.Pv', 'AIT201.Pv', 'LIT301.Pv']
ts_labels  = ['LIT101 — Raw Water Tank Level',
              'FIT101 — Inflow Rate',
              'AIT201 — Water Quality',
              'LIT301 — UF Feed Tank Level']

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Time Series — Key Process Sensors',
             fontsize=18, fontweight='bold', y=1.02, color='white')

for idx, (sensor, label, ax) in enumerate(zip(ts_sensors, ts_labels, axes.flat)):
    colour = PALETTE[idx % len(PALETTE)]
    vals = pd.to_numeric(
        df_raw[sensor].replace(r'(?i)^bad\s*input$', np.nan, regex=True),
        errors='coerce'
    )
    # Subsample for performance (plot every 10th point)
    step = max(1, len(vals) // 5000)
    ax.plot(ts.iloc[::step], vals.iloc[::step], color=colour, linewidth=0.6, alpha=0.9)
    ax.set_title(label, fontsize=13, fontweight='bold', color=colour)
    ax.set_xlabel('Time', fontsize=11)
    ax.set_ylabel('Value', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

---
## 7 · Actuator State Analysis

Actuators (`.Status` columns) typically take discrete values:  
- **1** → OFF  
- **2** → ON  

We compute the fraction of time each actuator spends in the ON state.

In [ ]:
# ─── Section 7: Actuator States ──────────────────────────────────────
# Convert status columns to numeric
df_status = df_raw[status_cols].copy()
for col in df_status.columns:
    df_status[col] = pd.to_numeric(
        df_status[col].replace(r'(?i)^bad\s*input$', np.nan, regex=True),
        errors='coerce'
    )

# Percentage of time each actuator is ON (Status == 2)
on_pct = (df_status == 2).sum() / df_status.notna().sum() * 100
off_pct = 100 - on_pct

# Sort by ON percentage
on_pct_sorted = on_pct.sort_values(ascending=True)
off_pct_sorted = off_pct[on_pct_sorted.index]

fig, ax = plt.subplots(figsize=(14, 10))

y_pos = range(len(on_pct_sorted))
ax.barh(y_pos, on_pct_sorted.values, color='#00d4ff', alpha=0.85, label='ON (Status=2)', height=0.7)
ax.barh(y_pos, off_pct_sorted.values, left=on_pct_sorted.values,
        color='#ff6b6b', alpha=0.65, label='OFF (Status=1)', height=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(on_pct_sorted.index, fontsize=9)
ax.set_xlabel('Percentage of Time (%)', fontsize=12)
ax.set_title('Actuator ON/OFF Duty Cycle', fontsize=16, fontweight='bold', pad=12)
ax.legend(fontsize=11, loc='lower right', framealpha=0.3)
ax.grid(True, axis='x', alpha=0.3, linestyle='--')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

---
## 8 · Sensor Correlation Heatmap

Full Pearson correlation matrix of all **continuous `.Pv` sensors** — reveals which physical measurements co-vary during normal operation. Strong correlations often reflect direct hydraulic or chemical coupling between sub-processes.

In [ ]:
# ─── Section 8: Correlation Heatmap ─────────────────────────────────
# Use the already-cleaned numeric .Pv data
corr_matrix = df_numeric.corr(method='pearson')

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.1f',
    annot_kws={'size': 5},
    linewidths=0.3,
    linecolor='#1a1a2e',
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    ax=ax,
)
ax.set_title('Sensor Correlation Matrix — Continuous (.Pv) Features',
             fontsize=16, fontweight='bold', pad=14)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.show()

---
## 9 · Top Correlated Sensor Pairs

Identify the 20 most strongly correlated sensor pairs (|r| > 0.7). These relationships are critical for:
- **Graph Neural Network** edge construction
- Understanding which sensors would move together during normal vs. attack conditions

In [ ]:
# ─── Section 9: Top Correlated Pairs ─────────────────────────────────
# Extract upper-triangle pairs
pairs = []
for i, col_a in enumerate(corr_matrix.columns):
    for j, col_b in enumerate(corr_matrix.columns):
        if j > i:  # upper triangle only
            r = corr_matrix.iloc[i, j]
            if abs(r) >= 0.7:
                pairs.append((col_a, col_b, round(r, 4)))

pairs_df = pd.DataFrame(pairs, columns=['Sensor A', 'Sensor B', 'Pearson r'])
pairs_df['|r|'] = pairs_df['Pearson r'].abs()
pairs_df = pairs_df.sort_values('|r|', ascending=False).head(20).reset_index(drop=True)

print(f'🔗 Top {len(pairs_df)} Most Correlated Sensor Pairs  (|r| ≥ 0.7)\n')
print(pairs_df[['Sensor A', 'Sensor B', 'Pearson r']].to_string(index=False))
print(f'\nTotal pairs with |r| ≥ 0.7: {len([p for p in pairs if abs(p[2]) >= 0.7])}')
print('\n💡 Physical significance:')
print('   • Highly correlated sensor pairs often share the same sub-process')
print('   • Level sensors (LIT) in connected tanks tend to be anti-correlated')
print('     (one fills while the other drains)')
print('   • Flow sensors (FIT) feeding the same tank correlate with its level sensor')
print('   • These edges form the adjacency matrix for our GNN-based anomaly detector')

---
## 10 · Process State Distribution

The SWaT testbed has 6 sub-processes (P1–P6). Each has a discrete state column (`P1_STATE` … `P6_STATE`) indicating the operational mode.

In [ ]:
# ─── Section 10: Process State Distribution ──────────────────────────
df_states = df_raw[state_cols].copy()
for col in df_states.columns:
    df_states[col] = pd.to_numeric(
        df_states[col].replace(r'(?i)^bad\s*input$', np.nan, regex=True),
        errors='coerce'
    )

# Compute value counts per state column (normalised as %)
all_states = sorted(set(int(v) for col in state_cols for v in df_states[col].dropna().unique()))

state_pct = pd.DataFrame(index=state_cols, columns=all_states, dtype=float)
for col in state_cols:
    vc = df_states[col].value_counts(normalize=True) * 100
    for s in all_states:
        state_pct.loc[col, s] = vc.get(s, 0.0)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(state_cols))
bar_width = 0.6
bottom = np.zeros(len(state_cols))

colours_for_states = PALETTE + ['#e0e0e0', '#4ecdc4', '#f38181', '#3a86ff']

for idx_s, state_val in enumerate(all_states):
    values = state_pct[state_val].values.astype(float)
    colour = colours_for_states[idx_s % len(colours_for_states)]
    ax.bar(x, values, bar_width, bottom=bottom, color=colour,
           alpha=0.85, label=f'State {state_val}', edgecolor='#1a1a2e', linewidth=0.5)
    # Annotate percentages > 5%
    for xi, v in zip(x, values):
        if v > 5:
            ax.text(xi, bottom[xi] + v / 2, f'{v:.0f}%',
                    ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    bottom += values

ax.set_xticks(x)
ax.set_xticklabels(state_cols, fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Process State Distribution (P1–P6)', fontsize=16, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper right', framealpha=0.3, ncol=min(len(all_states), 4))
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

---
## 11 · HAI Dataset Overview

The **HIL-based Augmented ICS Security** dataset provides a complementary testbed with **attack labels**, making it useful for supervised model evaluation.  
We load the training file to inspect its structure and compare with SWaT.

In [ ]:
# ─── Section 11: HAI Dataset Overview ────────────────────────────────
HAI_DIR = Path('data/hai')
hai_train_path = HAI_DIR / 'train1.csv.gz'
hai_test_path  = HAI_DIR / 'test1.csv.gz'

print('📂 Loading HAI dataset...\n')

hai_train = pd.read_csv(hai_train_path, compression='gzip')
hai_test  = pd.read_csv(hai_test_path, compression='gzip')

print(f'HAI Train : {hai_train.shape[0]:>9,} rows × {hai_train.shape[1]} cols')
print(f'HAI Test  : {hai_test.shape[0]:>9,} rows × {hai_test.shape[1]} cols')
print(f'HAI Total : {hai_train.shape[0] + hai_test.shape[0]:>9,} rows\n')

# Identify attack label columns
attack_cols = [c for c in hai_test.columns if c.startswith('attack')]
print(f'Attack label columns: {attack_cols}\n')

# Attack label distribution in test set
print('🚨 Attack Label Distribution (Test Set):')
print('-' * 45)
for ac in attack_cols:
    vc = hai_test[ac].value_counts()
    total = len(hai_test)
    for label, count in vc.items():
        pct = count / total * 100
        status = '🔴 ATTACK' if label == 1 else '🟢 Normal'
        print(f'  {ac} = {label} ({status}): {count:>8,}  ({pct:5.2f}%)')

In [ ]:
# ─── HAI vs SWaT Column Comparison ───────────────────────────────────
hai_cols = set(hai_train.columns)
swat_cols_set = set(df_raw.columns)

# Classify HAI columns
hai_pv     = [c for c in hai_train.columns if '.PV' in c or '.Pv' in c or c.endswith('_PV')]
hai_status = [c for c in hai_train.columns if '.Status' in c or c.endswith('_STATUS')]
hai_attack = [c for c in hai_train.columns if c.startswith('attack')]
hai_other  = [c for c in hai_train.columns if c not in set(hai_pv + hai_status + hai_attack)]

comparison = pd.DataFrame({
    'Attribute': ['Total Columns', 'Total Rows', 'Sensor/PV Columns',
                  'Actuator/Status Columns', 'Attack Labels', 'Has Timestamps',
                  'Domain'],
    'SWaT A9': [len(df_raw.columns), f'{len(df_raw):,}', len(pv_cols),
                len(status_cols), 0, 'Yes (t_stamp)', 'Water Treatment'],
    'HAI': [len(hai_train.columns), f'{len(hai_train) + len(hai_test):,}',
            len(hai_pv), len(hai_status), len(hai_attack),
            'Yes (timestamp)' if 'timestamp' in hai_train.columns.str.lower() else 'No',
            'HIL Augmented ICS'],
})

print('\n📊 SWaT vs HAI — Structural Comparison\n')
print(comparison.to_string(index=False))

In [ ]:
# ─── HAI Attack Distribution Bar Chart ───────────────────────────────
fig, axes = plt.subplots(1, len(attack_cols), figsize=(5 * len(attack_cols), 5))
if len(attack_cols) == 1:
    axes = [axes]

for idx, (ac, ax) in enumerate(zip(attack_cols, axes)):
    vc = hai_test[ac].value_counts().sort_index()
    colours = ['#6bcb77' if v == 0 else '#ff6b6b' for v in vc.index]
    bars = ax.bar(vc.index.astype(str), vc.values, color=colours, alpha=0.85,
                  edgecolor='#1a1a2e', linewidth=0.5, width=0.5)
    ax.set_title(ac, fontsize=14, fontweight='bold', color=PALETTE[idx % len(PALETTE)])
    ax.set_xlabel('Label', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.grid(True, axis='y', alpha=0.3, linestyle='--')
    # Annotate counts
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + h * 0.02,
                f'{int(h):,}', ha='center', va='bottom', fontsize=10, color='white')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Normal (0)', 'Attack (1)'], fontsize=10)

fig.suptitle('HAI Test Set — Attack Label Distribution',
             fontsize=16, fontweight='bold', y=1.04, color='white')
plt.tight_layout()
plt.show()

---
## 12 · Key Findings Summary

### 📋 Dataset Characteristics
- **SWaT A9** contains ~87K rows of **clean (normal) operation** data across 87 columns.
- Columns are cleanly categorised: ~31 continuous sensors (`.Pv`), ~32 actuators (`.Status`), 15 alarms, 2 speed, 6 process states.
- **`Bad Input`** string values appear in several sensor/actuator columns — these must be cleaned (→ NaN → forward fill) before modelling.

### 🔍 Sensor Behaviour
- Level sensors (LIT) show **cyclic fill/drain patterns** — classic normal water-treatment behaviour.
- Flow sensors (FIT) are often correlated with the downstream tank level they feed.
- Several sensor pairs have |r| > 0.7, forming natural **edges in the sensor graph** for GNN-based detection.

### 🔧 Actuator Patterns
- Most actuators have a dominant state (ON or OFF for >80% of time), reflecting the steady-state nature of normal operations.
- Some actuators are essentially always ON — potential candidates for constant-column removal.

### 🧬 Process States
- P1–P6 states are heavily concentrated in 1–2 dominant values, consistent with stable normal operation.
- Rare state transitions may be important for context-aware anomaly detection.

### 🆚 HAI Comparison
- HAI provides **labelled attack data** (attack_P1, P2, P3) that SWaT A9 lacks — critical for supervised evaluation.
- Column structures differ (HAI uses different naming conventions) but both capture sensor + actuator + state variables.
- RAKSHAK-ICS will train unsupervised on SWaT-normal and validate with HAI attack labels.

### 🚀 Next Steps
1. Run `src/preprocess.py` pipeline — clean → scale → sliding windows → graph construction
2. Train LSTM-Autoencoder on normal SWaT data
3. Build sensor correlation graph for GNN module
4. Evaluate anomaly detection on HAI attack segments